# REM-P2: estado nutricional en población infantil en control

**Exploración preliminar de la candidata 7 · 2020–2024**  
**Taller:** W1 — *Discovering questions worth investigating*  
**Propósito del notebook:** convertir la idea en una pregunta investigable, inspeccionar las bases y diccionarios disponibles, hacer controles iniciales de calidad y dejar una primera descripción reproducible.

> **Alcance:** esta exploración se refiere a recuentos administrativos de niños y niñas registrados en control REM-P2. No estima prevalencia poblacional, no sigue personas entre cortes y no permite inferir causalidad.

**Brief y rúbrica:** [W1_IIB423T-1_Brief_and_Rubric_Tomas_Fontecilla.pdf](../W1_IIB423T-1_Brief_and_Rubric_Tomas_Fontecilla.pdf). Esta notebook desarrolla la candidata 7. Si el entregable W1 del equipo debe comparar dos alternativas, falta añadir la comparación documentada con la otra candidata; no se considera resuelta aquí.


## 1. Planteamiento de negocio en los siete pasos del taller

| Paso | Propuesta inicial para REM-P2 |
|---|---|
| 1. Decisión | Proponer una vista de monitoreo para que el equipo de salud infantil identifique variaciones en los recuentos nutricionales registrados y priorice revisión de datos o preguntas de gestión. |
| 2. Criterio de éxito | Confirmar códigos y grupos de edad con el diccionario de cada año; cuantificar cobertura de registros emparejados; presentar cortes comparables con advertencias; generar al menos una visualización interpretable y reproducible. |
| 3. Pregunta de negocio | Entre niños y niñas registrados en control en REM-P2, ¿cómo varía la proporción de los recuentos `P2070503` (sobrepeso/riesgo de obesidad) y `P2070504` (obeso) sobre `P2060000` (`Col01`, ambos sexos) entre junio y diciembre de 2020–2024, y qué patrones por edad/sexo se observan en los campos desglosados que sí están reportados? |
| 4. Proceso | Control de salud infantil y registro REM-P2 de población en control/estado nutricional en cortes informados. |
| 5. Entidad, evento y estado | La base contiene recuentos agregados por servicio, establecimiento, corte (`Mes`) y código de prestación. No contiene filas de individuos. Un niño podría volver a contarse en otro corte. |
| 6. Representación | Una fila por combinación reportada de año, corte, servicio, establecimiento y código; `Col01` es total por ambos sexos y `Col04`–`Col31` detallan edad/sexo para los tramos de la hoja P2. Los campos de edad pueden estar vacíos y se auditan antes de usarlos. |
| 7. Medición | Métrica principal: `Col01` de `P2070503 + P2070504` dividido por `Col01` de `P2060000`, solo en unidades establecimiento-corte con denominador y ambos códigos observados. El desglose por edad/sexo es secundario y se informa con cobertura de campos disponibles. Ambos son **proporciones de recuentos registrados en control**, nunca prevalencias infantiles. |

**Uso potencial:** monitoreo de distribución de los registros por edad/sexo, comparación de cortes y detección de problemas de reporte. El MINSAL tiene una estrategia nacional 2023–2030 sobre sobrepeso y obesidad en niñez y adolescencia, que da contexto de relevancia, pero no convierte este indicador administrativo en una estimación poblacional: [Estrategia MINSAL 2023–2030](https://www.minsal.cl/lanzamiento-estrategia-para-detener-la-aceleracion-del-sobrepeso-y-obesidad-en-la-ninez-y-adolescencia-2023-2030/).


## 2. Fuentes disponibles y primera inspección

El inventario del proyecto señala que los archivos REM y sus diccionarios deben revisarse año a año porque códigos, columnas o secciones pueden cambiar. Esta notebook lee **un ZIP Serie P por año (2020–2024)** directamente, sin extraer ni alterar los originales. La copia CSV/TXT de 2024 que ya está descomprimida no se vuelve a contar.

Archivos locales de referencia:

- Inventario y procedencia: `FUENTES.csv` y `README.md` en la raíz del proyecto.
- Datos: `REM/2020/SERIE_REM_2020.zip` hasta `REM/2024/SERIE_REM_2024.zip`.
- Diccionarios: cada ZIP contiene el libro anual de códigos Serie P, hoja `P2`.
- Referencia DEIS de acceso a datos abiertos: [deis.minsal.cl](https://deis.minsal.cl/#datosabiertos).

### Hallazgos del primer escaneo (cálculo preliminar)

| Año | Filas de la Serie P completa | Filas de códigos P2 relevantes | Meses encontrados en esos códigos | Filas crudas `P2060000` por corte (antes de retirar duplicados) |
|---|---:|---:|---|---|
| 2020 | 455.729 | 9.944 | junio y diciembre | junio: 83; diciembre: 1.950 |
| 2021 | 890.014 | 19.088 | junio y diciembre | junio: 1.964; diciembre: 1.982 |
| 2022 | 993.701 | 19.787 | junio y diciembre | junio: 1.993; diciembre: 2.010 |
| 2023 | 1.117.209 | 20.539 | junio y diciembre | junio: 1.992; diciembre: 2.002 |
| 2024 | 1.200.514 | 20.843 | junio y diciembre | junio: 2.014; diciembre: 2.019 |

La cobertura registrada en junio de 2020 merece investigación: `P2060000` suma 51.788 en `Col01`, frente a 545.388–609.364 en junio de 2021–2024, y solo hay 83 registros establecimiento-servicio con denominador. No se atribuye esta diferencia a una causa sin confirmación documental. Además, los campos por edad/sexo (`Col04`–`Col31`) presentan blancos frecuentes; por eso `Col01` será la métrica central y el perfil por edad se reportará con su disponibilidad, sin tratar esos blancos como cero de forma implícita. En junio de 2024 solo 713 de las 1.764 unidades emparejadas tienen completas las 26 celdas de edad/sexo en denominador y ambas categorías; en diciembre son 727 de 1.762. En diciembre de 2021 se encontraron 18 filas involucradas en 9 claves duplicadas en dos establecimientos; sus valores son idénticos en las columnas analizadas, por lo que la rutina elimina las copias exactas y deja registro del control.

Los cinco diccionarios mantienen los códigos de población en control, categorías `P2070501`–`P2070506` y los tramos de edad de `Col04`–`Col31`. El código `P2501800` (sin evaluación por curso de vida o condición especial) aparece desde 2023. Por eso una reconciliación que lo incluya se presenta solo para 2023–2024 y requiere confirmar la regla aplicable en el manual de esos años.

En el primer cálculo de la métrica principal (`Col01`, ambos sexos), usando registros establecimiento-servicio-corte que reportan denominador y ambos códigos objetivo, se obtuvieron estos valores. Las cifras se reproducen en la celda de resumen; diciembre de 2021 se calcula después de quitar copias exactas:

| Año | Junio | Diciembre |
|---|---:|---:|
| 2020 | 33,61 %* | 35,05 % |
| 2021 | 35,46 % | 37,53 % |
| 2022 | 36,59 % | 35,46 % |
| 2023 | 34,59 % | 34,76 % |
| 2024 | 35,16 % | 34,72 % |

\* Junio de 2020 tiene una cobertura de establecimientos muy inferior. No debe interpretarse como corte nacional comparable sin aclarar primero la cobertura. Estos porcentajes son resúmenes de registros emparejados, no prevalencias ni cambios en niños individuales. El desglose por edad/sexo tiene blancos frecuentes: se analizará aparte con una tabla de cobertura y una sensibilidad que trata blancos como cero, no como una decisión confirmada. Las celdas siguientes reproducen el barrido y muestran cómo se construyen las cifras.


## 3. Preparación del entorno

Dependencias de análisis: `pandas` y `openpyxl`. Jupyter aporta IPython para mostrar la gráfica SVG generada más adelante. Si el kernel seleccionado no las tiene, activa el entorno del curso o instala las dependencias en ese entorno antes de ejecutar el notebook. No se necesita instalar paquetes para leer los CSV/TXT del ZIP.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import pandas as pd
import numpy as np
from openpyxl import load_workbook

# Busca la raíz tanto si Jupyter inicia desde la carpeta del proyecto como desde REM/.
_candidatos = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (p for p in _candidatos if (p / "REM" / "2020").is_dir()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No encuentro REM/2020. Abre el notebook desde la carpeta del proyecto "
        "o una carpeta descendiente."
    )
REM_DIR = PROJECT_ROOT / "REM"

P2_CODES = {"P2060000", *(f"P20705{i:02d}" for i in range(1, 7)), "P2501800"}
KEY_FIELDS = ["Mes", "IdServicio", "IdEstablecimiento", "CodigoPrestacion"]
UNIT_FIELDS = ["AnioFuente", "Mes", "IdServicio", "IdEstablecimiento"]
NUMERIC_FIELDS = [f"Col{i:02d}" for i in range(1, 32)]
AGE_SEX_MAP = [
    ("1 mes", "Hombres", "Col06"), ("1 mes", "Mujeres", "Col07"),
    ("2 meses", "Hombres", "Col08"), ("2 meses", "Mujeres", "Col09"),
    ("3 meses", "Hombres", "Col10"), ("3 meses", "Mujeres", "Col11"),
    ("4 meses", "Hombres", "Col12"), ("4 meses", "Mujeres", "Col13"),
    ("5 meses", "Hombres", "Col14"), ("5 meses", "Mujeres", "Col15"),
    ("6 meses", "Hombres", "Col16"), ("6 meses", "Mujeres", "Col17"),
    ("7–11 meses", "Hombres", "Col18"), ("7–11 meses", "Mujeres", "Col19"),
    ("12–17 meses", "Hombres", "Col20"), ("12–17 meses", "Mujeres", "Col21"),
    ("18–23 meses", "Hombres", "Col22"), ("18–23 meses", "Mujeres", "Col23"),
    ("24–35 meses", "Hombres", "Col24"), ("24–35 meses", "Mujeres", "Col25"),
    ("36–41 meses", "Hombres", "Col26"), ("36–41 meses", "Mujeres", "Col27"),
    ("42–47 meses", "Hombres", "Col28"), ("42–47 meses", "Mujeres", "Col29"),
    ("48–59 meses", "Hombres", "Col30"), ("48–59 meses", "Mujeres", "Col31"),
]
AGE_FIELDS_1_59 = [col for _, _, col in AGE_SEX_MAP]
print("Raíz del proyecto:", PROJECT_ROOT)
print("pandas:", pd.__version__)


## 4. Carga reproducible de los cinco ZIP

La lectura es por bloques y conserva solo los códigos P2 necesarios y las columnas `Col01`–`Col31`. Se mantienen los identificadores como texto para evitar conversiones de códigos. Se registra el número de filas de la Serie P completa y de las filas P2 seleccionadas.


In [ ]:
def localizar_miembros(zip_obj):
    miembros = [n for n in zip_obj.namelist() if not n.endswith("/")]
    datos = [n for n in miembros if Path(n).name.lower().startswith("seriep")
             and Path(n).suffix.lower() in {".txt", ".csv"}]
    diccionarios = [n for n in miembros if "sp_" in Path(n).name.lower()
                    and Path(n).suffix.lower() in {".xlsm", ".xlsx"}]
    if len(datos) != 1:
        raise ValueError(f"Esperaba un único archivo SerieP; encontré {datos}")
    if len(diccionarios) != 1:
        raise ValueError(f"Esperaba un diccionario SP; encontré {diccionarios}")
    return datos[0], diccionarios[0]

zip_por_anio = {}
for carpeta in sorted(REM_DIR.iterdir()):
    if carpeta.is_dir() and carpeta.name.isdigit():
        zips = sorted(carpeta.glob("*.zip"))
        if zips:
            zip_por_anio[int(carpeta.name)] = zips[0]

anios_esperados = set(range(2020, 2025))
if set(zip_por_anio) != anios_esperados:
    raise FileNotFoundError(
        f"Se esperaban ZIP REM para {sorted(anios_esperados)}; encontré {sorted(zip_por_anio)}"
    )

partes_p2 = []
filas_fuente = []
columnas_datos = ["Mes", "IdServicio", "IdEstablecimiento", "CodigoPrestacion"] + NUMERIC_FIELDS

for anio, ruta_zip in sorted(zip_por_anio.items()):
    with ZipFile(ruta_zip) as zf:
        miembro_datos, miembro_diccionario = localizar_miembros(zf)
        with zf.open(miembro_datos) as f:
            encabezado = pd.read_csv(f, sep=";", nrows=0, encoding="utf-8-sig")
        faltantes = set(columnas_datos) - set(encabezado.columns)
        if faltantes:
            raise ValueError(f"{anio}: faltan columnas requeridas: {sorted(faltantes)}")

        n_total = 0
        n_p2 = 0
        meses = set()
        partes_anio = []
        with zf.open(miembro_datos) as f:
            for bloque in pd.read_csv(
                f,
                sep=";",
                usecols=columnas_datos,
                dtype={
                    "IdServicio": "string",
                    "IdEstablecimiento": "string",
                    "CodigoPrestacion": "string",
                },
                chunksize=200_000,
                low_memory=False,
                encoding="utf-8-sig",
            ):
                n_total += len(bloque)
                bloque["Mes"] = pd.to_numeric(bloque["Mes"], errors="coerce").astype("Int64")
                bloque["CodigoPrestacion"] = bloque["CodigoPrestacion"].str.strip()
                seleccionado = bloque.loc[bloque["CodigoPrestacion"].isin(P2_CODES)].copy()
                if not seleccionado.empty:
                    n_p2 += len(seleccionado)
                    meses.update(seleccionado["Mes"].dropna().astype(int).tolist())
                    seleccionado["AnioFuente"] = anio
                    partes_anio.append(seleccionado)
        if partes_anio:
            partes_p2.extend(partes_anio)
        filas_fuente.append({
            "Año": anio,
            "ZIP": str(ruta_zip.relative_to(PROJECT_ROOT)),
            "Archivo dentro del ZIP": miembro_datos,
            "Diccionario dentro del ZIP": miembro_diccionario,
            "Filas Serie P": n_total,
            "Filas P2 seleccionadas": n_p2,
            "Meses en códigos seleccionados": ", ".join(map(str, sorted(meses))),
        })

datos_p2 = pd.concat(partes_p2, ignore_index=True)
for col in NUMERIC_FIELDS:
    datos_p2[col] = pd.to_numeric(datos_p2[col], errors="coerce")

inventario = pd.DataFrame(filas_fuente).sort_values("Año").reset_index(drop=True)
display(inventario)
print(f"Filas P2 seleccionadas cargadas: {len(datos_p2):,}")


## 5. Contrastar códigos y etiquetas con cada diccionario anual

Los rótulos no se heredan de un solo año: esta celda lee la hoja `P2` del diccionario incluido en cada ZIP. También permite advertir si aparece un cambio de código/etiqueta antes de comparar años.


In [ ]:
CODIGOS_A_REVISAR = ["P2060000", "P2070501", "P2070502", "P2070503",
                       "P2070504", "P2070505", "P2070506", "P2501800"]
filas_diccionario = []

for anio, ruta_zip in sorted(zip_por_anio.items()):
    with ZipFile(ruta_zip) as zf:
        _, miembro_diccionario = localizar_miembros(zf)
        wb = load_workbook(BytesIO(zf.read(miembro_diccionario)), read_only=True, data_only=True)
        ws = wb["P2"]
        encontrados = set()
        for row in ws.iter_rows(values_only=True):
            codigo = str(row[0]).strip() if row and row[0] is not None else ""
            if codigo in CODIGOS_A_REVISAR:
                partes = [" ".join(str(x).strip().split()) for x in row[1:3] if x is not None and str(x).strip()]
                filas_diccionario.append({"Año": anio, "Código": codigo,
                                          "Etiqueta en diccionario": " — ".join(partes)})
                encontrados.add(codigo)
        wb.close()

diccionario_p2 = pd.DataFrame(filas_diccionario)
display(diccionario_p2.sort_values(["Código", "Año"]))

### Mapa de columnas de edad y sexo

Los diccionarios P2 2020–2024 conservan el mapa de la tabla: `Col04`–`Col05` = menor de 1 mes; `Col06`–`Col31` = edades de 1 a 59 meses, por grupo y sexo. La métrica principal usa `Col01` (total por ambos sexos) e incluye el universo registrado en el formulario P2. El desglose de edades de 1–59 meses es secundario: calcula la disponibilidad de cada celda por edad/sexo y presenta una sensibilidad para blancos. No sumar `Col01` de los cinco años como si fueran niños únicos.


In [ ]:
mapa_edad = pd.DataFrame(AGE_SEX_MAP, columns=["Grupo de edad", "Sexo", "Columna"])
display(mapa_edad)


## 6. Controles iniciales de calidad (familias de control para mapear a las Five Cs de clase)

El brief pide usar las **Five Cs**. Para no inventar los nombres exactos de la versión usada en Clase 03, esta notebook deja los controles observables y el equipo debe mapearlos a esos nombres antes de entregar:

- **Cobertura/completitud:** años, cortes, registros por código, campos clave y fracción del denominador incluida al emparejar códigos.
- **Consistencia:** `Col01` frente a `Col02 + Col03` y frente a suma de grupos de edad cuando los campos están completos.
- **Conformidad/validez:** códigos presentes en el diccionario del año, valores negativos o no numéricos.
- **Unicidad:** clave `año + mes + servicio + establecimiento + código`; las copias exactas se deduplican en memoria y se informan. Los duplicados con valores distintos se excluyen del resumen y se listan.
- **Comparabilidad y actualidad:** mismos códigos/etiquetas y tramos entre años; cortes realmente disponibles; limitación especial de junio de 2020; fecha/versión de los archivos del paquete.

Estos son nombres descriptivos de los chequeos; no sustituyen la terminología literal de la cátedra.


In [ ]:
# Controles de coherencia para códigos relevantes. Los campos de edad vacíos
# se cuentan como faltantes; no se convierten aquí en ceros.
comparables = datos_p2.copy()
sex_cols = ["Col02", "Col03"]
age_cols_all = [f"Col{i:02d}" for i in range(4, 32)]
filas_calidad = []

for (anio, codigo), g in comparables.groupby(["AnioFuente", "CodigoPrestacion"], dropna=False):
    clave_completa = g[["Mes", "IdServicio", "IdEstablecimiento", "CodigoPrestacion"]].notna().all(axis=1)
    sex_completo = g[["Col01", *sex_cols]].notna().all(axis=1)
    suma_edades_observadas = g[age_cols_all].sum(axis=1, min_count=1)
    total_y_edad_observada = g["Col01"].notna() & suma_edades_observadas.notna()
    filas_calidad.append({
        "Año": int(anio),
        "Código": codigo,
        "Filas": len(g),
        "Claves incompletas": int((~clave_completa).sum()),
        "Col01 ausente": int(g["Col01"].isna().sum()),
        "Celdas negativas Col01–Col31": int((g[NUMERIC_FIELDS] < 0).sum().sum()),
        "Filas con algún blanco Col04–Col31": int(g[age_cols_all].isna().any(axis=1).sum()),
        "Col01≠Col02+Col03 (campos completos)": int(
            (((g["Col01"] - g["Col02"] - g["Col03"]).abs() > 0.001)
             .where(sex_completo, False)).sum()
        ),
        "Col01≠suma de celdas de edad informadas": int(
            (((g["Col01"] - suma_edades_observadas).abs() > 0.001)
             .where(total_y_edad_observada, False)).sum()
        ),
    })

calidad_inicial = pd.DataFrame(filas_calidad).sort_values(["Año", "Código"])
display(calidad_inicial)


## 7. Duplicados: registro, deduplicación analítica y conflictos

La deduplicación se aplica solo en memoria y solo a filas idénticas en la clave y las columnas que esta exploración analiza. Los archivos originales no se modifican. Si una clave tiene más de una versión distinta, se excluye toda la unidad establecimiento-corte afectada para evitar sumar filas ambiguas.


In [ ]:
CLAVE_CODIGO = UNIT_FIELDS + ["CodigoPrestacion"]
n_unidades_con_clave_incompleta = int(datos_p2[UNIT_FIELDS].isna().any(axis=1).sum())
datos_con_claves = datos_p2.loc[datos_p2[UNIT_FIELDS].notna().all(axis=1)].copy()
conteo_claves = (datos_con_claves.groupby(CLAVE_CODIGO, dropna=False)
                 .size().rename("n_filas").reset_index())
duplicados_clave = conteo_claves.loc[conteo_claves["n_filas"] > 1].copy()

columnas_identidad_analitica = CLAVE_CODIGO + NUMERIC_FIELDS
n_copias_exactas_eliminadas = int(
    datos_con_claves.duplicated(subset=columnas_identidad_analitica, keep="first").sum()
)
datos_sin_copias = datos_con_claves.drop_duplicates(
    subset=columnas_identidad_analitica, keep="first"
).copy()

claves_aun_duplicadas = (datos_sin_copias.groupby(CLAVE_CODIGO, dropna=False)
                         .size().loc[lambda s: s > 1].reset_index(name="n_versiones"))
if len(claves_aun_duplicadas):
    conflicto_unidades = claves_aun_duplicadas[UNIT_FIELDS].drop_duplicates()
    _idx = pd.MultiIndex.from_frame(conflicto_unidades)
    _idx_datos = pd.MultiIndex.from_frame(datos_sin_copias[UNIT_FIELDS])
    datos_limpios = datos_sin_copias.loc[~_idx_datos.isin(_idx)].copy()
else:
    conflicto_unidades = pd.DataFrame(columns=UNIT_FIELDS)
    datos_limpios = datos_sin_copias.copy()

display(duplicados_clave.sort_values(CLAVE_CODIGO))
print("Filas con clave incompleta excluidas:", n_unidades_con_clave_incompleta)
print("Filas exactas repetidas retiradas del análisis:", n_copias_exactas_eliminadas)
print("Unidades establecimiento-corte con conflictos excluidas:", len(conflicto_unidades))


## 8. Construir panel por establecimiento y corte

Se emparejan las filas por año, mes, servicio y establecimiento. La medida principal usa únicamente unidades que tienen denominador `P2060000` y filas observadas para ambos códigos `P2070503` y `P2070504`. La ausencia de una fila no se convierte automáticamente en cero.


In [ ]:
def tabla_codigo(codigo, campos):
    t = datos_limpios.loc[
        datos_limpios["CodigoPrestacion"].eq(codigo), UNIT_FIELDS + campos
    ].copy()
    t = t.set_index(UNIT_FIELDS)
    # Las claves duplicadas/conflictivas ya fueron retiradas arriba.
    t = t.rename(columns={c: f"{codigo}__{c}" for c in campos})
    return t

panel = tabla_codigo("P2060000", NUMERIC_FIELDS)
panel["tiene_P2060000"] = True
for codigo in ["P2070503", "P2070504", *[f"P20705{i:02d}" for i in (1,2,5,6)], "P2501800"]:
    t = tabla_codigo(codigo, NUMERIC_FIELDS)
    panel = panel.join(t, how="left")
    panel[f"tiene_{codigo}"] = panel.index.isin(t.index)

panel_filas = panel.reset_index()
campos_den_1_59 = [f"P2060000__{c}" for c in AGE_FIELDS_1_59]
campos_sobrepeso = [f"P2070503__{c}" for c in AGE_FIELDS_1_59]
campos_obesidad = [f"P2070504__{c}" for c in AGE_FIELDS_1_59]
campos_tres_codigos = ["P2060000__Col01", "P2070503__Col01", "P2070504__Col01"]


## 9. Resumen principal por corte y sensibilidad a códigos ausentes

La métrica principal usa `Col01` y unidades con denominador y ambos códigos objetivo presentes. Se muestran dos lecturas: **completa**, que no asume que una fila de código ausente equivale a cero; y **código ausente = cero (sensibilidad)**, que completa con cero solo una fila de código inexistente, conservando como faltante un `Col01` vacío dentro de una fila existente. Confirmar el significado de filas ausentes con DEIS antes de elegir una lectura definitiva.


In [ ]:
resumen_cortes = []
for (anio, mes), g in panel_filas.groupby(["AnioFuente", "Mes"], dropna=False):
    den_col = g["P2060000__Col01"]
    den_ok = den_col.notna()
    n_den = int(den_ok.sum())
    total_den = float(den_col.loc[den_ok].sum())

    columnas_objetivo = ["P2070503__Col01", "P2070504__Col01"]
    campos_edad_tres_codigos = (
        [f"P2060000__{c}" for c in AGE_FIELDS_1_59]
        + [f"P2070503__{c}" for c in AGE_FIELDS_1_59]
        + [f"P2070504__{c}" for c in AGE_FIELDS_1_59]
    )
    filas_codigos = (
        den_ok & g["tiene_P2070503"] & g["tiene_P2070504"]
        & g[columnas_objetivo].notna().all(axis=1)
    )
    den_emparejado = float(den_col.loc[filas_codigos].sum())
    num_sobrepeso = float(g.loc[filas_codigos, "P2070503__Col01"].sum())
    num_obesidad = float(g.loc[filas_codigos, "P2070504__Col01"].sum())
    num_emparejado = num_sobrepeso + num_obesidad
    filas_edad_completa = filas_codigos & g[campos_edad_tres_codigos].notna().all(axis=1)

    # Sensibilidad: ausencia de fila de categoría = cero. Un Col01 vacío
    # dentro de una fila existente sigue siendo faltante.
    sobrepeso_cero = g["P2070503__Col01"].copy()
    obesidad_cero = g["P2070504__Col01"].copy()
    sobrepeso_cero.loc[~g["tiene_P2070503"]] = 0
    obesidad_cero.loc[~g["tiene_P2070504"]] = 0
    valido_cero = den_ok & sobrepeso_cero.notna() & obesidad_cero.notna()
    den_cero = float(den_col.loc[valido_cero].sum())
    num_cero = float(sobrepeso_cero.loc[valido_cero].sum() + obesidad_cero.loc[valido_cero].sum())

    resumen_cortes.append({
        "Año": int(anio),
        "Corte": int(mes),
        "Unidades establecimiento-servicio con P2060000": n_den,
        "Unidades con P206 y ambas categorías": int(filas_codigos.sum()),
        "Unidades con 26 campos edad/sexo completos": int(filas_edad_completa.sum()),
        "Cobertura de unidades emparejadas (%)": 100 * filas_codigos.sum() / n_den if n_den else np.nan,
        "P2060000 Col01": total_den,
        "P2060000 Col01 emparejado": den_emparejado,
        "Cobertura del denominador Col01 (%)": 100 * den_emparejado / total_den if total_den else np.nan,
        "P2070503 Col01 emparejado": num_sobrepeso,
        "P2070504 Col01 emparejado": num_obesidad,
        "Numerador combinado emparejado": num_emparejado,
        "Proporción emparejada (%)": 100 * num_emparejado / den_emparejado if den_emparejado else np.nan,
        "Proporción con código ausente=0 (%)": 100 * num_cero / den_cero if den_cero else np.nan,
    })

resumen_cortes = pd.DataFrame(resumen_cortes).sort_values(["Año", "Corte"]).reset_index(drop=True)
display(resumen_cortes.round(2))


### Interpretación preliminar

La proporción principal se mantiene alrededor de 33,6–37,5 % en los diez cortes inspeccionados; la tabla no demuestra una tendencia sostenida. Las diferencias junio/diciembre pueden reflejar el proceso de reporte y la composición de establecimientos, además del fenómeno de interés. Junio de 2020 es especialmente débil por su cobertura registrada. La columna de sensibilidad permite ver cuánto depende el resultado de interpretar una fila de código ausente como cero.

Para el informe, describe cada porcentaje como **proporción de recuentos registrados en control**. La misma persona podría estar en ambos cortes; el cambio no mide transición individual ni incidencia. El perfil por edad/sexo se interpreta según la disponibilidad de cada columna y se presenta como análisis secundario.


## 10. Desglose por grupo de edad y sexo

Los campos de edad/sexo tienen vacíos frecuentes. Para cada tramo/sexo, la lectura **solo celdas informadas** calcula la proporción en unidades que tienen denominador y ambos códigos presentes en esa columna; la lectura **blancos = cero** completa celdas vacías como cero entre unidades emparejadas. La segunda es una sensibilidad, no una imputación validada. Revisa número y porcentaje de unidades disponibles antes de interpretar diferencias por edad/sexo.


In [ ]:
detalle_edad_sexo = []
for (anio, mes), g in panel_filas.groupby(["AnioFuente", "Mes"], dropna=False):
    tiene_codigos = (
        g["P2060000__Col01"].notna()
        & g["tiene_P2070503"] & g["tiene_P2070504"]
        & g["P2070503__Col01"].notna() & g["P2070504__Col01"].notna()
    )
    for grupo, sexo, col in AGE_SEX_MAP:
        den_col = f"P2060000__{col}"
        sob_col = f"P2070503__{col}"
        obe_col = f"P2070504__{col}"
        valido = tiene_codigos & g[[den_col, sob_col, obe_col]].notna().all(axis=1)
        n_match = int(tiene_codigos.sum())
        n_disponibles = int(valido.sum())
        den_informado = float(g.loc[valido, den_col].sum())
        num_informado = float(g.loc[valido, sob_col].sum() + g.loc[valido, obe_col].sum())

        # Sensibilidad únicamente: tratar celdas de edad/sexo vacías como cero.
        den_cero = float(g.loc[tiene_codigos, den_col].fillna(0).sum())
        num_cero = float(
            g.loc[tiene_codigos, sob_col].fillna(0).sum()
            + g.loc[tiene_codigos, obe_col].fillna(0).sum()
        )
        detalle_edad_sexo.append({
            "Año": int(anio), "Corte": int(mes), "Grupo de edad": grupo, "Sexo": sexo,
            "Unidades con códigos objetivo": n_match,
            "Unidades con campos de edad informados": n_disponibles,
            "Cobertura del campo (%)": 100 * n_disponibles / n_match if n_match else np.nan,
            "Denominador en campos informados": den_informado,
            "Numerador en campos informados": num_informado,
            "Proporción entre campos informados (%)": 100 * num_informado / den_informado if den_informado else np.nan,
            "Proporción sensibilidad blancos=0 (%)": 100 * num_cero / den_cero if den_cero else np.nan,
        })

detalle_edad_sexo = pd.DataFrame(detalle_edad_sexo).sort_values(
    ["Año", "Corte", "Grupo de edad", "Sexo"]
).reset_index(drop=True)
display(detalle_edad_sexo.loc[detalle_edad_sexo["Año"].eq(2024)].round(2))


## 11. Visualización de proporción emparejada por corte

La gráfica usa la métrica principal `Col01` y una escala vertical de 0 a 100 % para evitar exagerar variaciones pequeñas. Junio y diciembre son dos cortes semestrales; no hay observaciones mensuales intermedias en estos códigos. La marca de junio de 2020 debe leerse con la advertencia de cobertura anterior.


In [ ]:
from IPython.display import SVG, display


def grafico_lineas_svg(tabla, columna="Proporción emparejada (%)"):
    años = sorted(tabla["Año"].dropna().astype(int).unique())
    ancho, alto = 820, 470
    izq, der, arriba, abajo = 76, 28, 42, 86
    x0, x1 = izq, ancho - der
    y0, y1 = arriba, alto - abajo
    colores = {6: "#147D79", 12: "#BD6B28"}

    def x_pos(i):
        return x0 if len(años) == 1 else x0 + i * (x1 - x0) / (len(años) - 1)
    def y_pos(valor):
        return y1 - (float(valor) / 100.0) * (y1 - y0)

    partes = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{ancho}" height="{alto}" viewBox="0 0 {ancho} {alto}">',
        '<rect width="100%" height="100%" fill="white"/>',
        '<text x="76" y="24" font-family="sans-serif" font-size="16" font-weight="600">Proporción de recuentos registrados, Col01</text>',
    ]
    for tick in (0, 20, 40, 60, 80, 100):
        y = y_pos(tick)
        partes.append(f'<line x1="{x0}" y1="{y:.1f}" x2="{x1}" y2="{y:.1f}" stroke="#d9dee5" stroke-width="1"/>')
        partes.append(f'<text x="{x0-12}" y="{y+4:.1f}" text-anchor="end" font-family="sans-serif" font-size="11" fill="#445">{tick}%</text>')
    for i, año in enumerate(años):
        x = x_pos(i)
        partes.append(f'<text x="{x:.1f}" y="{y1+24}" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#334">{año}</text>')
    partes.append(f'<line x1="{x0}" y1="{y0}" x2="{x0}" y2="{y1}" stroke="#68717d"/>')
    partes.append(f'<line x1="{x0}" y1="{y1}" x2="{x1}" y2="{y1}" stroke="#68717d"/>')

    for mes, nombre in ((6, "Junio"), (12, "Diciembre")):
        serie = tabla.loc[tabla["Corte"].eq(mes)].sort_values("Año")
        puntos = [(int(fila["Año"]), fila[columna]) for _, fila in serie.iterrows() if pd.notna(fila[columna])]
        if puntos:
            coords = " ".join(f"{x_pos(años.index(a)):.1f},{y_pos(v):.1f}" for a, v in puntos)
            partes.append(f'<polyline points="{coords}" fill="none" stroke="{colores[mes]}" stroke-width="2.5"/>')
            for a, v in puntos:
                x, y = x_pos(años.index(a)), y_pos(v)
                partes.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="5" fill="{colores[mes]}"/>')
        lx = x0 + (mes == 12) * 130
        partes.append(f'<line x1="{lx}" y1="{alto-35}" x2="{lx+22}" y2="{alto-35}" stroke="{colores[mes]}" stroke-width="3"/>')
        partes.append(f'<text x="{lx+28}" y="{alto-31}" font-family="sans-serif" font-size="12" fill="#334">{nombre}</text>')
    partes.append(f'<text x="{x0}" y="{alto-10}" font-family="sans-serif" font-size="11" fill="#667">Junio de 2020 tiene cobertura de registros muy baja.</text>')
    partes.append('</svg>')
    return SVG("".join(partes))

# Esta gráfica presenta la lectura de casos completos; revisar también la sensibilidad.
display(grafico_lineas_svg(resumen_cortes))


## 12. Reconciliación exploratoria de categorías (2023–2024)

En 2023 y 2024 el diccionario agrega `P2501800` para niños sin evaluación nutricional por curso de vida o condición especial. Se compara `P2060000 Col01` con la suma de `P2070501`–`P2070506` y `P2501800 Col01`. Para que la comparación sea calculable en cada establecimiento-corte, las filas de código ausentes se tratan como cero **solo en este chequeo de sensibilidad**. Una diferencia no prueba por sí sola error de reporte: la regla debe confirmarse en el manual DEIS de la versión anual y con el significado de filas omitidas.


In [ ]:
CODIGOS_RECONCILIACION = [f"P20705{i:02d}" for i in range(1, 7)] + ["P2501800"]
filas_reconciliacion = []

for anio in (2023, 2024):
    den = tabla_codigo("P2060000", ["Col01"])
    den = den.loc[den.index.get_level_values("AnioFuente") == anio].copy()
    den_col = "P2060000__Col01"
    base = den.copy()
    status_fields = []
    for codigo in CODIGOS_RECONCILIACION:
        t = tabla_codigo(codigo, ["Col01"])
        t = t.loc[t.index.get_level_values("AnioFuente") == anio]
        base = base.join(t, how="left")
        presente = pd.Series(base.index.isin(t.index), index=base.index)
        base[f"tiene_{codigo}"] = presente
        status_fields.append(f"{codigo}__Col01")

    base = base.reset_index()
    for mes, g in base.groupby("Mes"):
        den_ok = g[den_col].notna()
        categorias = g[status_fields].copy()
        presentes = [f"tiene_{c}" for c in CODIGOS_RECONCILIACION]
        for codigo, campo in zip(CODIGOS_RECONCILIACION, status_fields):
            categorias.loc[~g[f"tiene_{codigo}"], campo] = 0
        categorias_completas = categorias.notna().all(axis=1)
        suma_categorias = categorias.sum(axis=1, min_count=len(status_fields))
        diferencia = suma_categorias - g[den_col]
        exacto = den_ok & categorias_completas & diferencia.abs().le(0.001)
        filas_reconciliacion.append({
            "Año": anio,
            "Corte": int(mes),
            "Unidades con denominador": int(den_ok.sum()),
            "Exactamente reconciliadas": int(exacto.sum()),
            "Reconciliación exacta (%)": 100 * exacto.sum() / den_ok.sum() if den_ok.sum() else np.nan,
            "Suma categorías menos P2060000": float(diferencia.loc[den_ok & categorias_completas].sum()),
            "Filas de categoría ausentes (conteo)": int((~g[[f"tiene_{c}" for c in CODIGOS_RECONCILIACION]].all(axis=1) & den_ok).sum()),
        })

reconciliacion = pd.DataFrame(filas_reconciliacion).sort_values(["Año", "Corte"])
display(reconciliacion.round(2))


## 13. Interpretación, alcance y límites

- **Unidad de análisis:** agregado establecimiento-corte-código de prestación. Los registros no permiten seguir a niños individuales ni saber si el mismo niño figura en varios establecimientos o cortes.
- **Denominador:** `P2060000 Col01`, población en control reportada en REM-P2. La proporción compara categorías nutricionales con ese registro administrativo; no con la población infantil total de Chile o de una comuna.
- **Cortes:** para los códigos P2 seleccionados se hallan junio y diciembre en cada año. No existe una serie mensual para modelar una trayectoria mes a mes.
- **Cobertura:** junio de 2020 destaca por bajo número de establecimientos y recuento en control; examinarlo con DEIS antes de incluirlo en una comparación histórica.
- **Datos faltantes:** filas de códigos ausentes pueden representar cero o falta de reporte; las dos lecturas se separan en las tablas. Solicitar confirmación. Los blancos en campos de edad/sexo también son frecuentes; no se imputan en la lectura principal y se informa disponibilidad por tramo.
- **Reconciliación:** comparar categorías con el total es un diagnóstico inicial. El código `P2501800` aparece desde 2023; no aplicar esa suma hacia atrás. Confirmar manuales y reglas exactas de 2023 y 2024.
- **Cambios de clasificación:** las hojas P2 anuales muestran códigos objetivo y grupos de edad estables en esta primera revisión; mantener el diccionario anual como autoridad y volver a validar cualquier ampliación.
- **Geografía:** hay códigos de región/comuna en la Serie P, pero esta investigación no depende de cartografía. Cualquier comparación geográfica sería descriptiva, debe validar los códigos y no equivale a tasas locales sin denominadores poblacionales pertinentes.
- **Causalidad y significancia:** este barrido es descriptivo; no identifica causas ni prueba diferencias estadísticas.

El manual DEIS [REM Serie P 2025–2026](https://repositoriodeis.minsal.cl/ContenidoSitioWeb2020/REM/2025/SERIE/MANUAL_REM_P_2025_V1.1.pdf) puede ayudar a entender la estructura general, pero es posterior a las observaciones de 2020–2024. Para declarar una regla oficial de reconciliación, usar el manual aplicable a cada año.


## 14. Ajuste a la rúbrica W1: potencial y trabajo pendiente

### Futuro uso de ML

Con cinco años y solo dos cortes anuales, hay como máximo diez puntos semestrales por serie; no hay base suficiente para una predicción robusta de ML. Primero se requiere confirmar cobertura, filas ausentes, reglas de reconciliación y comparabilidad. Si se consigue una serie histórica más granular y armonizada, podría explorarse un pronóstico de recuentos registrados y compararlo con una línea base simple, evaluando fuera de muestra. La predicción no sería de prevalencia.

### Futuro dashboard

Es viable un panel de monitoreo con filtros por año/corte, edad, sexo, establecimiento y eventualmente región; métricas de recuentos y proporción registrada; número de establecimientos emparejados; y alertas de cobertura, duplicados y reconciliación. El título y las etiquetas deben decir “registros en control” y no “prevalencia”.

### Preguntas concretas para el instructor

1. En estos archivos, ¿una fila de código ausente equivale a cero o indica falta de reporte? ¿Cuál es la versión del manual REM-P2 que debe aplicarse a cada año?
2. ¿La comparación histórica de junio/diciembre debe excluir o tratar aparte junio de 2020 por su baja cobertura de establecimientos?
3. ¿Es aceptable presentar como indicador exploratorio la proporción de `P2070503 + P2070504 Col01` sobre `P2060000 Col01`, siempre rotulada como proporción de recuentos registrados en control y dejando el desglose de edad/sexo como secundario?

### Bitácora de proceso (completar con el equipo)

| Elemento requerido | Registro del equipo |
|---|---|
| Decisiones de alcance y motivo | Completar durante la reunión del equipo. |
| Controles ejecutados, resultados y cambios | Notebook reproducible; anotar fecha de ejecución y cualquier exclusión adicional. |
| Retroalimentación real del instructor y respuesta del equipo | Pendiente; no completar hasta recibirla. |
| Contribución de cada integrante | Completar con tareas efectivamente realizadas. |
| Reflexión individual de cada integrante | Pendiente de cada integrante. |
| Uso de IA y verificación | Codex apoyó el inventario de archivos, comparación inicial de diccionarios y construcción del flujo. El equipo debe ejecutar la notebook, cotejar las cifras con los ZIP/diccionarios y documentar toda corrección antes de entregar. |

**Checklist antes de entregar W1:** incluir la segunda candidata y la comparación exigida por la rúbrica; verificar Five Cs con los nombres exactos vistos en clase; registrar fuentes, versión/fecha de recuperación y resultados reproducidos; añadir la retroalimentación, contribuciones y reflexiones reales del equipo.
